# 01 · COCO 2017 val: download, clean, EDA

The steps are:
1. Download `captions_val2017.json`.
2. Clean the captions and build the manifest.
3. Download the 5,000 val images and cache them as 512×512 center-crops on Drive.
4. Run EDA.

Every step skips itself if its output already exists. The first full run is mostly the 778 MB image download.

In [ ]:
# --- Setup: mount Drive, clone or update the repo, install pinned deps. Safe to rerun. ---
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Pranav1011/ie7615-generative-project.git"
BRANCH = "main"
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

def _git(*args, url=None):
    """Run git. For a private repo, add a GITHUB_TOKEN Colab secret (key icon in the sidebar)."""
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    auth_url = url.replace("https://", f"https://x-access-token:{token}@") if (url and token) else url
    cmd = ["git", *[auth_url if a == "<URL>" else a for a in args]]
    res = subprocess.run(cmd, capture_output=True, text=True)
    out = (res.stdout + res.stderr).replace(token, "***") if token else res.stdout + res.stderr
    if res.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/ie7615-generative-project")
    if (REPO_DIR / ".git").exists():
        print(_git("-C", str(REPO_DIR), "pull", "--ff-only", "<URL>", BRANCH, url=REPO_URL))
    else:
        _git("clone", "-q", "-b", BRANCH, "<URL>", str(REPO_DIR), url=REPO_URL)
        _git("-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL)  # never persist a token
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("repo:", REPO_DIR, "| commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
from src.utils import load_config, get_paths, set_seed
cfg = load_config()
paths = get_paths(cfg)
set_seed(cfg["seed"])
print("artifact root:", paths["root"])

## 1. Captions → cleaned manifest

In [ ]:
from src import data, preprocess

data.download_coco_val(cfg, paths, images=False)
records, stats = data.prepare_manifest(cfg, paths)
stats

## 2. Images → 512×512 cache (on Drive)

In [ ]:
missing = preprocess.count_uncached(records, paths["root"])
print(f"{missing} of {len(records)} images not cached yet")
if missing:
    data.download_coco_val(cfg, paths, images=True)  # zip kept on Drive, extracted to local scratch
    print(preprocess.cache_resized(records, paths["images"], paths["root"],
                                   size=cfg["image"]["size"], quality=cfg["image"]["jpeg_quality"]))

In [ ]:
# Sanity check: one cached image -> model tensor in [-1, 1]
ds = preprocess.CocoImageCaptionDataset(records, paths["root"], size=cfg["image"]["size"])
item = ds[0]
print(len(ds), "items |", tuple(item["pixel_values"].shape),
      f"range [{item['pixel_values'].min():.2f}, {item['pixel_values'].max():.2f}] |", item["caption"])

## 3. EDA

In [ ]:
import matplotlib.pyplot as plt

df = data.manifest_to_frame(records)
print(df["n_words"].describe().round(2))
print("captions per image:", df.groupby("image_id").size().value_counts().sort_index().to_dict())

eda_dir = paths["outputs"] / "eda"
eda_dir.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(df["n_words"], bins=range(0, df["n_words"].max() + 2), color="#3b6ea5")
axes[0].axvline(cfg["data"]["min_words"], ls="--", c="grey"); axes[0].axvline(cfg["data"]["max_words"], ls="--", c="grey")
axes[0].set(title="Caption length (words), after filtering", xlabel="words", ylabel="captions")
wh =[(r["width"], r["height"]) for r in records]
axes[1].scatter([w for w, _ in wh], [h for _, h in wh], s=3, alpha=0.3, color="#3b6ea5")
axes[1].set(title="Original image size", xlabel="width", ylabel="height")
plt.tight_layout(); fig.savefig(eda_dir / "caption_length_and_sizes.png", dpi=120); plt.show()

In [ ]:
from collections import Counter
words = Counter(w.strip(".,!?;:'\"") for c in df["lower"] for w in c.split())
stop = {"a", "an", "the", "of", "on", "in", "with", "and", "is", "to", "are", "at", "next", "two", "some", "his", "her", "its", "that", "for", "it", "up", "by", "there", "while", "has", "from"}
print([w for w, _ in words.most_common(60) if w not in stop][:30])

In [ ]:
import random
from PIL import Image
from IPython.display import display
from src.utils import make_image_grid

rng = random.Random(cfg["seed"])
sample = rng.sample(records, 9)
grid = make_image_grid([Image.open(preprocess.cached_path(r, paths["root"])) for r in sample],
                       [r["captions"][0]["clean"] for r in sample], ncols=3, cell_size=256)
grid.save(eda_dir / "sample_grid.png")
display(grid)